# FRAME — Fractionation & Mixing Estimation

**FRAME** uses Markov Chain Monte Carlo (MCMC) to model the mixing of stable isotopes, optionally accounting for isotopic fractionation.

This notebook demonstrates a simple 2-D mixing example (two isotope ratios, three end-member sources) using the public repository at [github.com/malewick/frame](https://github.com/malewick/frame).

---

## 1 — Setup

Clone the repository and make sure the required libraries are available (they are all pre-installed in Colab).

In [ ]:
# Clone the FRAME repository (skip if already cloned)
import os
if not os.path.isdir('frame'):
    !git clone https://github.com/malewick/frame.git frame
else:
    print('Repository already present — skipping clone.')

# Move into the repo root so relative paths work correctly
os.chdir('frame')
print('Working directory:', os.getcwd())

In [ ]:
# All heavy dependencies (numpy, pandas, matplotlib, scipy, sympy) come
# pre-installed in Colab.  Only sympy sometimes needs a version bump.
!pip install -q --upgrade sympy

# Add the source directory to the Python path
import sys
sys.path.insert(0, 'src')

# Use the non-interactive Agg backend so plots render inline
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
%matplotlib inline

import NDimModel
print('FRAME imported successfully.')

## 2 — Input data

The `input/` directory ships several example datasets.  Here we use the simplest one:

| File | Description |
|---|---|
| `generic2D_data.csv` | One mixture sample with two isotope measurements and their uncertainties |
| `generic2D_sources_delta.csv` | Three end-member sources, each defined by their isotope values and spread |

In [ ]:
import pandas as pd

data_file    = 'input/generic2D_data.csv'
sources_file = 'input/generic2D_sources_delta.csv'

print('=== Mixture data ===')
display(pd.read_csv(data_file))

print('\n=== End-member sources ===')
display(pd.read_csv(sources_file))

## 3 — Run the MCMC model

Key parameters:

| Parameter | Meaning |
|---|---|
| `niter` | Maximum number of MCMC iterations (terminates early if `chain_length` is reached) |
| `burnout` | Burn-in steps discarded before recording the chain |
| `chain_length` | Number of accepted steps to record |

Setting `plotting_switch = False` disables the real-time interactive window; the model will still generate and save all diagnostic plots at the end.

In [ ]:
model = NDimModel.Model()

model.load_from_fielnames(
    data_file    = data_file,
    sources_file = sources_file,
    aux_file     = '',
)

model.set_outdir(out_dir='output')
model.set_iterations(
    niter            = 1_000_000,
    burnout          = 100,
    max_chain_entries= 500,
)

model.myfmt          = 'png'
model.plotting_switch = False   # no interactive window in Colab

model.run_model()

## 4 — Results

### 4a — Numerical summary

In [ ]:
import glob

results_csv = glob.glob('output/**/*_results.csv', recursive=True)
for f in results_csv:
    print(f'Results file: {f}')
    display(pd.read_csv(f))

### 4b — Diagnostic plots

FRAME saves three types of plots per group:

| Suffix | Content |
|---|---|
| `_TimeSeries_` | MCMC chain traces for each variable |
| `_Path_` | Trajectory of accepted steps on the isotope map |
| `_CorrelationPlot_` | Posterior histograms and pairwise correlations |

In [ ]:
from IPython.display import Image, display as ipy_display

plot_files = sorted(glob.glob('output/**/*.png', recursive=True))

if not plot_files:
    print('No PNG plots found — the run may have saved in a different format.')
else:
    for pf in plot_files:
        print(pf)
        ipy_display(Image(pf))

## 6 — Fractionation example with a custom auxiliary-variable prior

When an auxiliary variable `r` (e.g. a residual fraction or reduction progress) is included in
the model equation, FRAME samples it from a prior distribution.  The default is
`Uniform(0, 1)`, which matches the original paper.  You can override this with
`model.set_aux_prior()`.

| `prior_type` | Formula | When to use |
|---|---|---|
| `'uniform'` | `r ~ Uniform[r_min, r_max]` | Default; appropriate when `r` is expected anywhere in the range with equal probability |
| `'loguniform'` | `log(r) ~ Uniform[log(r_min), log(r_max)]` | When `r` spans multiple orders of magnitude (requires `r_min > 0`) |

In [ ]:
## 6a — 2-D mixing with Rayleigh-type fractionation (uniform prior, default)

data_file_frac    = 'input/fractionation2D_data.csv'
sources_file_frac = 'input/generic2D_sources_delta.csv'
aux_file_frac     = 'input/fractionation2D_frac.csv'

import NDimModel
model_frac = NDimModel.Model()

model_frac.load_from_fielnames(
    data_file    = data_file_frac,
    sources_file = sources_file_frac,
    aux_file     = aux_file_frac,
)

# Default prior: r ~ Uniform[0, 1]  (same as original model behaviour)
# This line is optional — shown here for clarity.
model_frac.set_aux_prior('r', prior_type='uniform', r_min=0.0, r_max=1.0)

model_frac.set_outdir(out_dir='output/fractionation_uniform')
model_frac.set_iterations(niter=1_000_000, burnout=100, max_chain_entries=500)
model_frac.myfmt          = 'png'
model_frac.plotting_switch = False

model_frac.run_model()
print('Done — uniform prior run.')

In [ ]:
## 6b — Same model, log-uniform prior for r

# A log-uniform prior is appropriate when r is expected to be small (close to 0)
# but could in principle reach 1 — e.g. a reduction fraction that is usually
# below 10 % but occasionally near 100 %.  It places equal weight per decade,
# preventing the sampler from over-representing large values.
#
# r_min must be > 0 for the log-uniform prior.

model_frac2 = NDimModel.Model()

model_frac2.load_from_fielnames(
    data_file    = data_file_frac,
    sources_file = sources_file_frac,
    aux_file     = aux_file_frac,
)

model_frac2.set_aux_prior('r', prior_type='loguniform', r_min=0.001, r_max=1.0)

model_frac2.set_outdir(out_dir='output/fractionation_loguniform')
model_frac2.set_iterations(niter=1_000_000, burnout=100, max_chain_entries=500)
model_frac2.myfmt          = 'png'
model_frac2.plotting_switch = False

model_frac2.run_model()
print('Done — log-uniform prior run.')

In [ ]:
## 6c — Compare results from both prior choices

import pandas as pd, glob

for label, out_dir in [('uniform prior', 'output/fractionation_uniform'),
                       ('log-uniform prior', 'output/fractionation_loguniform')]:
    files = glob.glob(f'{out_dir}/**/*_results.csv', recursive=True)
    for f in files:
        print(f'\n=== {label} — {f} ===')
        display(pd.read_csv(f))

from IPython.display import Image, display as ipy_display
for label, out_dir in [('uniform prior', 'output/fractionation_uniform'),
                       ('log-uniform prior', 'output/fractionation_loguniform')]:
    pngs = sorted(glob.glob(f'{out_dir}/**/*.png', recursive=True))
    print(f'\n--- {label} ---')
    for p in pngs:
        ipy_display(Image(p))

## 5 — Try other examples

Swap `data_file` / `sources_file` (and optionally `aux_file`) for any of the files in `input/`:

| Dataset | Data file | Sources file | Aux file |
|---|---|---|---|
| Generic 2-D | `generic2D_data.csv` | `generic2D_sources_delta.csv` | — |
| 2-D with fractionation | `fractionation2D_data.csv` | `generic2D_sources_delta.csv` | `fractionation2D_frac.csv` |
| 2-D measurement on edge | `edge2D_data.csv` | `generic2D_sources_delta.csv` | — |
| Water isotopes (real data) | `real world examples/H2O_data.csv` | `real world examples/H2O_sources.csv` | `real world examples/H2O_frac.csv` |
| Nitrous oxide (real data) | `real world examples/N2O_data.csv` | `real world examples/N2O_sources.csv` | `real world examples/N2Ofrac.csv` |